In [1]:
pip install wfdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 7.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import subprocess
import sys

print("Installing compatible protobuf version...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "protobuf==3.20.0", "-q"])
print("Done!")

# Sekarang restart kernel/notebook
print("\nKERNEL RESTART REQUIRED!")
print("Please restart the kernel now (Jupyter will prompt you)")

Installing compatible protobuf version...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 7.8 MB/s eta 0:00:00
Done!

KERNEL RESTART REQUIRED!
Please restart the kernel now (Jupyter will prompt you)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 3.20.0 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.0 which is incompatible.
google-cloud-secret-manager 2.25.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 3.20.0 which is incompatible.
google-cloud-vision 3.11.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 3.20.0 which is incompatible.
google-cloud-monitoring 2.28.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.

In [3]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import KFold
from scipy.signal import filtfilt, butter
import wfdb

2025-11-23 05:30:23.597332: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763875823.849344      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763875823.921273      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [11]:
# =====================================================
# 1. IMPROVED PREPROCESSING (Match GUI)
# =====================================================

def notch_filter_formula2(signal, f0=50, fs=128, r=0.98):
    """Notch filter 50Hz"""
    w0 = 2 * np.pi * f0 / fs
    c = np.cos(w0)
    b = np.array([1, -2*c, 1])
    a = np.array([1, -2*r*c, r*r])
    return filtfilt(b, a, signal)

def bandpass_filter(signal, low=0.5, high=50, fs=128, order=4):
    """Bandpass 0.5-50Hz (matches GUI training!)"""
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, signal)

def normalize_robust(signal):
    """Robust normalization using percentiles 1-99"""
    q1, q99 = np.percentile(signal, [1, 99])
    if q99 - q1 == 0:
        return np.zeros_like(signal)
    signal_clipped = np.clip(signal, q1, q99)
    return (signal_clipped - q1) / (q99 - q1)


In [ ]:
# =====================================================
# 2. LOAD SVDB WITH PVC SAMPLES (NEW!)
# =====================================================

def load_svdb_with_pvc(sve_symbols={"A", "a", "J", "S", "s"},
                       pvc_symbols={"V", "v","N"},
                       window=512,
                       fs=128,
                       mask_width=48,
                       random_state=42):
    """
    Load SVDB dengan:
    - SVE samples sebagai Class 1 (positive)
    - PVC + Normal samples sebagai Class 0 (negative)
    
    Model akan learn: "SVE vs non-SVE (termasuk PVC)"
    """
    rng = np.random.default_rng(random_state)
    
    svdb_records = [
        '800','801','802','803','804','805','806','807','808','809','811','812','820','821','822','823',
        '824','825','826','827','828','829','840','841','842','844','846','847','848','849','850','851',
        '852','853','854','855','856','857','858','859','860','861','862','863','864','865','866','867',
        '868','869','870','871','872','873','874','875','876','877','878','879','880','881','882','883',
        '884','891','892','893','894'
    ]

    X_sve = []
    y_sve = []
    X_non_sve = []  # Mix of PVC + Normal
    y_non_sve = []

    for rec in svdb_records:
        print(f"Loading {rec}... ", end="", flush=True)
        try:
            sig, fields = wfdb.rdsamp(f"/kaggle/input/svdbfix/{rec}")
            ann = wfdb.rdann(f"/kaggle/input/svdbfix/{rec}", "atr")
            ecg = sig[:, 1]

            # Preprocessing (match GUI!)
            ecg = notch_filter_formula2(ecg, 50, fs, r=0.98)
            ecg = bandpass_filter(ecg, 0.5, 50, fs, order=4)
            ecg = normalize_robust(ecg)

            # Create mask for SVE only
            mask = np.zeros(len(ecg))
            events = list(zip(ann.sample, ann.symbol))

            # Mark ONLY SVE symbols
            for samp, sym in events:
                if sym in sve_symbols:
                    st = max(0, samp - mask_width)
                    ed = min(len(ecg), samp + mask_width)
                    mask[st:ed] = 1

            total = len(ecg) // window
            for w in range(total):
                s = w * window
                e = s + window
                win_syms = {sym for samp, sym in events if s <= samp < e}

                # Check what symbols are present
                has_sve = any(sym in sve_symbols for sym in win_syms)
                has_pvc = any(sym in pvc_symbols for sym in win_syms)
                has_unknown = any(sym not in sve_symbols and sym not in pvc_symbols and sym != '' for sym in win_syms)

                # Skip if has unknown symbols
                if has_unknown:
                    continue

                win_sig = ecg[s:e]
                win_mask = mask[s:e]

                # IMPORTANT: If has SVE, go to SVE class
                if has_sve:
                    X_sve.append(win_sig)
                    y_sve.append(win_mask)
                # Otherwise (PVC or Normal), go to non-SVE class
                else:
                    X_non_sve.append(win_sig)
                    y_non_sve.append(win_mask)  # All zeros for non-SVE

            print("✓")
        except Exception as e:
            print(f"Error: {e}")

    X_sve = np.array(X_sve).reshape(-1, window, 1)
    y_sve = np.array(y_sve).reshape(-1, window, 1)
    X_non_sve = np.array(X_non_sve).reshape(-1, window, 1)
    y_non_sve = np.array(y_non_sve).reshape(-1, window, 1)

    print(f"\nDataset before balancing:")
    print(f"SVE samples: {X_sve.shape[0]}")
    print(f"Non-SVE (PVC+Normal) samples: {X_non_sve.shape[0]}")

    # Balance: keep same amount of SVE and non-SVE
    target = min(X_sve.shape[0], X_non_sve.shape[0])

    if X_sve.shape[0] > target:
        idx = rng.choice(X_sve.shape[0], target, replace=False)
        X_sve = X_sve[idx]
        y_sve = y_sve[idx]

    if X_non_sve.shape[0] > target:
        idx = rng.choice(X_non_sve.shape[0], target, replace=False)
        X_non_sve = X_non_sve[idx]
        y_non_sve = y_non_sve[idx]

    # Combine
    X_all = np.concatenate([X_sve, X_non_sve], 0)
    y_all = np.concatenate([y_sve, y_non_sve], 0)

    print(f"\nDataset after balancing:")
    print(f"SVE: {X_sve.shape[0]}")
    print(f"Non-SVE: {X_non_sve.shape[0]}")
    print(f"Total: {X_all.shape}")

    # Shuffle
    idx = rng.permutation(X_all.shape[0])
    return X_all[idx], y_all[idx]

In [13]:
# =====================================================
# 3. LOSS FUNCTIONS & METRICS
# =====================================================

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    """Dice loss"""
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) +
                                           tf.reduce_sum(y_pred_f) + smooth)

def weighted_focal_tversky_loss(y_true, y_pred,
                                alpha=0.6, beta=0.4, gamma=2.5, smooth=1e-6):
    """Weighted focal tversky loss"""
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    tp = tf.reduce_sum(y_true_f * y_pred_f)
    fp = tf.reduce_sum((1 - y_true_f) * y_pred_f)
    fn = tf.reduce_sum(y_true_f * (1 - y_pred_f))
    tversky = (tp + smooth) / (tp + alpha*fn + beta*fp + smooth)
    return tf.pow((1 - tversky), gamma)

def norm_accuracy(y_true, y_pred, threshold=0.5, smooth=1e-6):
    """Normalized accuracy"""
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]) > threshold, tf.float32)
    diff = y_pred_f - y_true_f
    norm = tf.norm(diff, ord=2)
    denom = tf.sqrt(tf.cast(tf.size(y_true_f), tf.float32))
    return 1.0 - (norm / (denom + smooth))

In [14]:
# =====================================================
# 4. BUILD UNET 1D MODEL
# =====================================================

from tensorflow import keras
from tensorflow.keras import layers, models

def build_unet_1d(input_length=512, dr=0.15):
    """UNet 1D architecture"""
    inputs = layers.Input((input_length, 1))
    
    # Encoder 1
    c1 = layers.Conv1D(32, 3, padding='same', activation='relu')(inputs)
    c1 = layers.BatchNormalization()(c1)
    c1 = layers.Dropout(dr)(c1)
    c1 = layers.Conv1D(32, 3, padding='same', activation='relu')(c1)
    c1 = layers.BatchNormalization()(c1)
    p1 = layers.MaxPooling1D(2)(c1)
    
    # Encoder 2
    c2 = layers.Conv1D(64, 3, padding='same', activation='relu')(p1)
    c2 = layers.BatchNormalization()(c2)
    c2 = layers.Dropout(dr)(c2)
    c2 = layers.Conv1D(64, 3, padding='same', activation='relu')(c2)
    c2 = layers.BatchNormalization()(c2)
    p2 = layers.MaxPooling1D(2)(c2)
    
    # Encoder 3
    c3 = layers.Conv1D(128, 3, padding='same', activation='relu')(p2)
    c3 = layers.BatchNormalization()(c3)
    c3 = layers.Dropout(dr)(c3)
    c3 = layers.Conv1D(128, 3, padding='same', activation='relu')(c3)
    c3 = layers.BatchNormalization()(c3)
    p3 = layers.MaxPooling1D(2)(c3)
    
    # Encoder 4
    c4 = layers.Conv1D(256, 3, padding='same', activation='relu')(p3)
    c4 = layers.BatchNormalization()(c4)
    c4 = layers.Dropout(dr)(c4)
    c4 = layers.Conv1D(256, 3, padding='same', activation='relu')(c4)
    c4 = layers.BatchNormalization()(c4)
    p4 = layers.MaxPooling1D(2)(c4)
    
    # Bottleneck
    bn = layers.Conv1D(512, 3, padding='same', activation='relu')(p4)
    bn = layers.BatchNormalization()(bn)
    bn = layers.Dropout(dr)(bn)
    bn = layers.Conv1D(512, 3, padding='same', activation='relu')(bn)
    bn = layers.BatchNormalization()(bn)
    
    # Decoder 1
    u4 = layers.UpSampling1D(2)(bn)
    u4 = layers.Concatenate()([u4, c4])
    d4 = layers.Conv1D(256, 3, padding='same', activation='relu')(u4)
    d4 = layers.BatchNormalization()(d4)
    d4 = layers.Dropout(dr)(d4)
    d4 = layers.Conv1D(256, 3, padding='same', activation='relu')(d4)
    d4 = layers.BatchNormalization()(d4)
    
    # Decoder 2
    u3 = layers.UpSampling1D(2)(d4)
    u3 = layers.Concatenate()([u3, c3])
    d3 = layers.Conv1D(128, 3, padding='same', activation='relu')(u3)
    d3 = layers.BatchNormalization()(d3)
    d3 = layers.Dropout(dr)(d3)
    d3 = layers.Conv1D(128, 3, padding='same', activation='relu')(d3)
    d3 = layers.BatchNormalization()(d3)
    
    # Decoder 3
    u2 = layers.UpSampling1D(2)(d3)
    u2 = layers.Concatenate()([u2, c2])
    d2 = layers.Conv1D(64, 3, padding='same', activation='relu')(u2)
    d2 = layers.BatchNormalization()(d2)
    d2 = layers.Dropout(dr)(d2)
    d2 = layers.Conv1D(64, 3, padding='same', activation='relu')(d2)
    d2 = layers.BatchNormalization()(d2)
    
    # Decoder 4
    u1 = layers.UpSampling1D(2)(d2)
    u1 = layers.Concatenate()([u1, c1])
    d1 = layers.Conv1D(32, 3, padding='same', activation='relu')(u1)
    d1 = layers.BatchNormalization()(d1)
    d1 = layers.Dropout(dr)(d1)
    d1 = layers.Conv1D(32, 3, padding='same', activation='relu')(d1)
    d1 = layers.BatchNormalization()(d1)
    
    # Output
    outputs = layers.Conv1D(1, 1, activation='sigmoid')(d1)
    
    return models.Model(inputs, outputs)

In [15]:
# =====================================================
# 5. TRAINING (50-FOLD CV)
# =====================================================

print("="*70)
print("LOADING SVDB DATASET WITH PVC SAMPLES")
print("="*70)
X, y = load_svdb_with_pvc()
print(f"\nDataset shape: {X.shape}, {y.shape}")

print("\n" + "="*70)
print("STARTING 50-FOLD CROSS VALIDATION WITH PVC TRAINING")
print("="*70)

kfold = KFold(n_splits=50, shuffle=True, random_state=42)
all_loss = []
all_dice = []
all_norm = []
all_prec = []
all_rec = []
all_f1 = []

fold = 1
for train_idx, val_idx in kfold.split(X):
    print(f"\nFOLD {fold}/50 | Train {len(train_idx)} | Val {len(val_idx)}")

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # Build model
    model = build_unet_1d(512, dr=0.15)

    # Compile
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss='binary_crossentropy',
        metrics=[
            dice_coefficient,
            norm_accuracy,
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )

    # Train
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=100,
        batch_size=32,
        verbose=1,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor='val_recall',
                patience=12,
                restore_best_weights=True,
                mode='max'),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=5,
                min_lr=1e-6)
        ]
    )

    # Evaluate
    results = model.evaluate(X_val, y_val, verbose=0)
    loss, dice, norm_acc, prec, rec = results
    f1 = 2 * (prec * rec) / (prec + rec + 1e-6)

    print(f"  Results: Dice={dice*100:.1f}% | NormAcc={norm_acc*100:.1f}% | "
          f"P={prec*100:.1f}% | R={rec*100:.1f}% | F1={f1*100:.1f}%")

    all_loss.append(loss)
    all_dice.append(dice)
    all_norm.append(norm_acc)
    all_prec.append(prec)
    all_rec.append(rec)
    all_f1.append(f1)

    # Save model
    model.save(f"unet_fold_{fold}_with_pvc.h5")
    print(f"  Saved: unet_fold_{fold}_with_pvc.h5")
    
    fold += 1

LOADING SVDB DATASET WITH PVC SAMPLES
Loading 800... ✓
Loading 801... ✓
Loading 802... ✓
Loading 803... ✓
Loading 804... ✓
Loading 805... ✓
Loading 806... ✓
Loading 807... ✓
Loading 808... ✓
Loading 809... ✓
Loading 811... ✓
Loading 812... ✓
Loading 820... ✓
Loading 821... ✓
Loading 822... ✓
Loading 823... ✓
Loading 824... ✓
Loading 825... ✓
Loading 826... ✓
Loading 827... ✓
Loading 828... ✓
Loading 829... ✓
Loading 840... ✓
Loading 841... ✓
Loading 842... ✓
Loading 844... ✓
Loading 846... ✓
Loading 847... ✓
Loading 848... ✓
Loading 849... ✓
Loading 850... ✓
Loading 851... ✓
Loading 852... ✓
Loading 853... ✓
Loading 854... ✓
Loading 855... ✓
Loading 856... ✓
Loading 857... ✓
Loading 858... ✓
Loading 859... ✓
Loading 860... ✓
Loading 861... ✓
Loading 862... ✓
Loading 863... ✓
Loading 864... ✓
Loading 865... ✓
Loading 866... ✓
Loading 867... ✓
Loading 868... ✓
Loading 869... ✓
Loading 870... ✓
Loading 871... ✓
Loading 872... ✓
Loading 873... ✓
Loading 874... ✓
Loading 875... ✓
Loading 87

I0000 00:00:1763875867.063673      48 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/100


I0000 00:00:1763875886.825312     127 service.cc:148] XLA service 0x78f74c03fda0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1763875886.826304     127 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1763875888.682187     127 cuda_dnn.cc:529] Loaded cuDNN version 90300


  7/395 ━━━━━━━━━━━━━━━━━━━━ 7s 20ms/step - dice_coefficient: 0.2236 - loss: 0.9567 - norm_accuracy: 0.2718 - precision: 0.1450 - recall: 0.5418  

...

  Results: Dice=68.3% | NormAcc=81.7% | P=83.6% | R=85.0% | F1=84.3%
  Saved: unet_fold_46_with_pvc.h5

FOLD 47/50 | Train 12629 | Val 257
Epoch 1/100
395/395 ━━━━━━━━━━━━━━━━━━━━ 52s 50ms/step - dice_coefficient: 0.2791 - loss: 0.5724 - norm_accuracy: 0.5069 - precision: 0.2895 - recall: 0.4978 - val_dice_coefficient: 0.0982 - val_loss: 0.4640 - val_norm_accuracy: 0.6037 - val_precision: 0.8699 - val_recall: 0.0159 - learning_rate: 5.0000e-04
Epoch 2/100
395/395 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - dice_coefficient: 0.4960 - loss: 0.2511 - norm_accuracy: 0.7161 - precision: 0.7685 - recall: 0.5667 - val_dice_coefficient: 0.6348 - val_loss: 0.2028 - val_norm_accuracy: 0.7394 - val_precision: 0.7676 - val_recall: 0.6432 - learning_rate: 5.0000e-04
Epoch 3/100
395/395 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - dice_coefficient: 0.6061 - loss: 0.1922 - norm_accuracy: 0.7423 - precision: 0.8130 - recall: 0.6593 - val_dice_coefficient: 0.7263 - val_loss: 0.1561 - val_norm_accuracy: 0.7790 - val_pre

  Results: Dice=85.8% | NormAcc=84.4% | P=90.3% | R=89.2% | F1=89.8%
  Saved: unet_fold_47_with_pvc.h5

FOLD 48/50 | Train 12629 | Val 257
Epoch 1/100
395/395 ━━━━━━━━━━━━━━━━━━━━ 47s 51ms/step - dice_coefficient: 0.2786 - loss: 0.5718 - norm_accuracy: 0.4997 - precision: 0.2830 - recall: 0.5003 - val_dice_coefficient: 0.2280 - val_loss: 0.4105 - val_norm_accuracy: 0.6109 - val_precision: 0.5745 - val_recall: 0.1315 - learning_rate: 5.0000e-04
Epoch 2/100
395/395 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - dice_coefficient: 0.5119 - loss: 0.2502 - norm_accuracy: 0.7155 - precision: 0.7757 - recall: 0.5863 - val_dice_coefficient: 0.6487 - val_loss: 0.2055 - val_norm_accuracy: 0.7437 - val_precision: 0.7342 - val_recall: 0.7242 - learning_rate: 5.0000e-04
Epoch 3/100
395/395 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - dice_coefficient: 0.6204 - loss: 0.1869 - norm_accuracy: 0.7450 - precision: 0.8150 - recall: 0.6794 - val_dice_coefficient: 0.6750 - val_loss: 0.1617 - val_norm_accuracy: 0.7520 - val_pre

  Results: Dice=74.8% | NormAcc=77.2% | P=74.0% | R=84.1% | F1=78.7%
  Saved: unet_fold_48_with_pvc.h5

FOLD 49/50 | Train 12629 | Val 257
Epoch 1/100
395/395 ━━━━━━━━━━━━━━━━━━━━ 51s 63ms/step - dice_coefficient: 0.2747 - loss: 0.5819 - norm_accuracy: 0.4895 - precision: 0.2722 - recall: 0.5017 - val_dice_coefficient: 0.1254 - val_loss: 0.4098 - val_norm_accuracy: 0.6618 - val_precision: 0.6892 - val_recall: 0.0363 - learning_rate: 5.0000e-04
Epoch 2/100
395/395 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - dice_coefficient: 0.5167 - loss: 0.2420 - norm_accuracy: 0.7180 - precision: 0.7704 - recall: 0.5966 - val_dice_coefficient: 0.5453 - val_loss: 0.2062 - val_norm_accuracy: 0.7449 - val_precision: 0.6714 - val_recall: 0.7434 - learning_rate: 5.0000e-04
Epoch 3/100
395/395 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - dice_coefficient: 0.6133 - loss: 0.1813 - norm_accuracy: 0.7481 - precision: 0.8147 - recall: 0.6788 - val_dice_coefficient: 0.6064 - val_loss: 0.1605 - val_norm_accuracy: 0.7842 - val_pre

  Results: Dice=67.4% | NormAcc=81.2% | P=82.0% | R=84.3% | F1=83.1%
  Saved: unet_fold_49_with_pvc.h5

FOLD 50/50 | Train 12629 | Val 257
Epoch 1/100
395/395 ━━━━━━━━━━━━━━━━━━━━ 49s 53ms/step - dice_coefficient: 0.2662 - loss: 0.5774 - norm_accuracy: 0.4906 - precision: 0.2634 - recall: 0.4713 - val_dice_coefficient: 0.0906 - val_loss: 0.4028 - val_norm_accuracy: 0.6905 - val_precision: 0.4432 - val_recall: 0.0120 - learning_rate: 5.0000e-04
Epoch 2/100
395/395 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - dice_coefficient: 0.5212 - loss: 0.2369 - norm_accuracy: 0.7212 - precision: 0.7803 - recall: 0.6014 - val_dice_coefficient: 0.5270 - val_loss: 0.2048 - val_norm_accuracy: 0.7403 - val_precision: 0.7518 - val_recall: 0.6202 - learning_rate: 5.0000e-04
Epoch 3/100
395/395 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - dice_coefficient: 0.6278 - loss: 0.1804 - norm_accuracy: 0.7492 - precision: 0.8298 - recall: 0.6786 - val_dice_coefficient: 0.5620 - val_loss: 0.1761 - val_norm_accuracy: 0.7631 - val_pre

  Results: Dice=72.1% | NormAcc=83.0% | P=87.8% | R=80.6% | F1=84.0%
  Saved: unet_fold_50_with_pvc.h5


In [16]:
# =====================================================
# 6. RESULTS SUMMARY
# =====================================================

print("\n" + "="*70)
print("TRAINING COMPLETE - FINAL RESULTS")
print("="*70)
print(f"\nAverage Dice Coefficient: {np.mean(all_dice)*100:.2f}% ± {np.std(all_dice)*100:.2f}%")
print(f"Average Norm Accuracy:    {np.mean(all_norm)*100:.2f}% ± {np.std(all_norm)*100:.2f}%")
print(f"Average Precision:        {np.mean(all_prec)*100:.2f}% ± {np.std(all_prec)*100:.2f}%")
print(f"Average Recall:           {np.mean(all_rec)*100:.2f}% ± {np.std(all_rec)*100:.2f}%")
print(f"Average F1 Score:         {np.mean(all_f1)*100:.2f}% ± {np.std(all_f1)*100:.2f}%")

# Select best model
best_fold = np.argmax(all_f1) + 1
print(f"\n✅ Best model: unet_fold_{best_fold}_with_pvc.h5")
print(f"   F1 Score: {all_f1[best_fold-1]*100:.2f}%")

print("\n✅ TRAINING DONE!")
print("Next: Load best model into GUI and test")


TRAINING COMPLETE - FINAL RESULTS

Average Dice Coefficient: 79.08% ± 5.67%
Average Norm Accuracy:    81.69% ± 2.07%
Average Precision:        85.61% ± 3.23%
Average Recall:           86.82% ± 2.45%
Average F1 Score:         86.18% ± 2.32%

✅ Best model: unet_fold_47_with_pvc.h5
   F1 Score: 89.77%

✅ TRAINING DONE!
Next: Load best model into GUI and test


In [21]:
# =====================================================
# CREATE FINAL MODEL
# =====================================================

print("\n" + "="*60)
print("CREATING FINAL MODEL")
print("="*60)

# Get ensemble predictions
print("\nGenerating pseudo-labels dari ensemble...")
y_ensemble = np.mean([model.predict(X, verbose=0) for model in models_ensemble], axis=0)
print(f"✓ Shape: {y_ensemble.shape}")

# Build model
print("\nBuilding UNet model...")
final_model = build_unet_1d(512)

final_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
    loss='binary_crossentropy',
    metrics=[dice_coefficient, norm_accuracy]
)

# Train
print("\nTraining...")
history = final_model.fit(
    X, y_ensemble,
    epochs=100,
    batch_size=128,
    shuffle=True,
    verbose=1
)

# Save
final_model.save("unet_svdb_FINAL.h5")
print("\n✓ Saved: unet_svdb_FINAL.h5")

# Test
print("\n" + "="*60)
print("TEST")
print("="*60)

test_size = min(200, X_val.shape[0])
X_test = X_val[:test_size]
y_test = tf.cast(y_val[:test_size], tf.float32)

pred = tf.cast(final_model.predict(X_test, verbose=0), tf.float32)
dice = dice_coefficient(y_test, pred).numpy()

print(f"\nFinal Model Dice: {dice*100:.2f}%")
print(f"✓ READY FOR INFERENCE!")

# Function
def predict_final_model(ecg_512):
    if ecg_512.ndim == 1:
        ecg_512 = ecg_512.reshape(-1, 1)
    X = ecg_512.reshape(1, 512, 1)
    return final_model.predict(X, verbose=0).squeeze()

print("\nUsage: pred = predict_final_model(X_test[0])")


CREATING FINAL MODEL

Generating pseudo-labels dari ensemble...
✓ Shape: (12886, 512, 1)

Building UNet model...

Training...
Epoch 1/100


2025-11-23 09:41:21.920754: E external/local_xla/xla/service/slow_operation_alarm.cc:65] Trying algorithm eng4{k11=2} for conv (f32[128,256,1,64]{3,2,1,0}, u8[0]{0}) custom-call(f32[128,256,1,64]{3,2,1,0}, f32[256,256,1,3]{3,2,1,0}), window={size=1x3 pad=0_0x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardInput", backend_config={"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"leakyrelu_alpha":0,"side_input_scale":0},"force_earliest_schedule":false,"operation_queue_id":"0","wait_on_operation_queues":[]} is taking a while...
2025-11-23 09:41:21.954247: E external/local_xla/xla/service/slow_operation_alarm.cc:133] The operation took 1.033619644s
Trying algorithm eng4{k11=2} for conv (f32[128,256,1,64]{3,2,1,0}, u8[0]{0}) custom-call(f32[128,256,1,64]{3,2,1,0}, f32[256,256,1,3]{3,2,1,0}), window={size=1x3 pad=0_0x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardInput", backend_config={"cudnn_conv_backend_confi

100/101 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - dice_coefficient: 0.2540 - loss: 0.6853 - norm_accuracy: 0.4136

E0000 00:00:1763890905.591116     125 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1763890905.877693     125 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1763890907.032402     125 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1763890907.287593     125 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


101/101 ━━━━━━━━━━━━━━━━━━━━ 69s 228ms/step - dice_coefficient: 0.2547 - loss: 0.6835 - norm_accuracy: 0.4158
Epoch 2/100
101/101 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - dice_coefficient: 0.4285 - loss: 0.3727 - norm_accuracy: 0.7307
Epoch 3/100
101/101 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - dice_coefficient: 0.5410 - loss: 0.2508 - norm_accuracy: 0.7805
Epoch 4/100
101/101 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - dice_coefficient: 0.6051 - loss: 0.2044 - norm_accuracy: 0.8013
Epoch 5/100
101/101 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - dice_coefficient: 0.6576 - loss: 0.1750 - norm_accuracy: 0.8145
Epoch 6/100
101/101 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - dice_coefficient: 0.6912 - loss: 0.1584 - norm_accuracy: 0.8260
Epoch 7/100
101/101 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - dice_coefficient: 0.7166 - loss: 0.1449 - norm_accuracy: 0.8354
Epoch 8/100
101/101 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - dice_coefficient: 0.7268 - loss: 0.1407 - norm_accuracy: 0.8415
Epoch 9/100
101/101 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/s


✓ Saved: unet_svdb_FINAL.h5

TEST

Final Model Dice: 84.23%
✓ READY FOR INFERENCE!

Usage: pred = predict_final_model(X_test[0])


In [ ]:
# =====================================================
# FINAL EVALUATION – RECORD-LEVEL TEST (NO LEAKAGE)
# =====================================================

import numpy as np
import tensorflow as tf
import wfdb
from scipy.signal import butter, filtfilt
import json

# =====================================================
# 1. PREPROCESSING (HARUS SAMA DENGAN TRAINING)
# =====================================================

def notch_filter_formula2(signal, f0=50, fs=128, r=0.98):
    w0 = 2 * np.pi * f0 / fs
    c = np.cos(w0)
    b = np.array([1, -2*c, 1])
    a = np.array([1, -2*r*c, r*r])
    return filtfilt(b, a, signal)

def bandpass_filter(signal, low=0.5, high=50, fs=128, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, signal)

def normalize_robust(signal):
    q1, q99 = np.percentile(signal, [1, 99])
    if q99 - q1 == 0:
        return np.zeros_like(signal)
    signal = np.clip(signal, q1, q99)
    return (signal - q1) / (q99 - q1)

# =====================================================
# 2. METRIC
# =====================================================

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (
        tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth
    )

# =====================================================
# 3. PILIH REKAMAN TEST (TIDAK PERNAH DIPAKAI TRAINING)
# =====================================================

TEST_RECORDS = [
    '885','886','887','889','890'
]

# =====================================================
# 4. LOAD TEST DATA (RECORD-LEVEL)
# =====================================================

def load_test_records(records,
                      window=512,
                      fs=128,
                      mask_width=48,
                      sve_symbols={"A","a","J","S","s"}):
    
    X_test, y_test = [], []

    for rec in records:
        print(f"Loading TEST record {rec}...")
        sig, _ = wfdb.rdsamp(f"/kaggle/input/svdbfix/{rec}")
        ann = wfdb.rdann(f"/kaggle/input/svdbfix/{rec}", "atr")

        ecg = sig[:, 1]

        # preprocessing
        ecg = notch_filter_formula2(ecg, 50, fs)
        ecg = bandpass_filter(ecg, 0.5, 50, fs)
        ecg = normalize_robust(ecg)

        # SVE mask
        mask = np.zeros(len(ecg))
        for samp, sym in zip(ann.sample, ann.symbol):
            if sym in sve_symbols:
                st = max(0, samp-mask_width)
                ed = min(len(ecg), samp+mask_width)
                mask[st:ed] = 1

        # windowing
        for w in range(len(ecg)//window):
            s = w * window
            e = s + window
            X_test.append(ecg[s:e])
            y_test.append(mask[s:e])

    X_test = np.array(X_test).reshape(-1, window, 1)
    y_test = np.array(y_test).reshape(-1, window, 1)

    print(f"\n✓ Total TEST windows: {X_test.shape[0]}")
    return X_test, y_test

# =====================================================
# 5. LOAD MODEL
# =====================================================

print("\n" + "="*60)
print("LOADING TRAINED MODEL")
print("="*60)

final_model = tf.keras.models.load_model(
    "/kaggle/input/svdbmodel/keras/default/1/unet_svdb_FINAL (1).h5",
    custom_objects={"dice_coefficient": dice_coefficient},
    compile=False
)

print("✓ Model loaded: unet_svdb_FINAL.h5")

# =====================================================
# 6. PREPARE TEST DATA
# =====================================================

print("\n" + "="*60)
print("PREPARING RECORD-LEVEL TEST DATA")
print("="*60)

X_test, y_test = load_test_records(TEST_RECORDS)
y_test = tf.cast(y_test, tf.float32)

# === BATASI JUMLAH WINDOW TEST ===
MAX_TEST_WINDOWS = 200
test_size = min(MAX_TEST_WINDOWS, X_test.shape[0])

X_test = X_test[:test_size]
y_test = y_test[:test_size]

print(f"✓ Using {test_size} test windows")

# =====================================================
# 7. RUN INFERENCE
# =====================================================

print("\n" + "="*60)
print("RUNNING INFERENCE")
print("="*60)

pred_probs = tf.cast(final_model.predict(X_test, verbose=0), tf.float32)
pred_binary = tf.cast(pred_probs > 0.5, tf.float32)

# =====================================================
# 8. CONFUSION MATRIX
# =====================================================

y_true = tf.reshape(y_test, [-1])
y_pred = tf.reshape(pred_binary, [-1])

tp = tf.reduce_sum(y_true * y_pred)
fp = tf.reduce_sum((1 - y_true) * y_pred)
fn = tf.reduce_sum(y_true * (1 - y_pred))
tn = tf.reduce_sum((1 - y_true) * (1 - y_pred))

# =====================================================
# 9. METRICS
# =====================================================

precision = tp / (tp + fp + 1e-7)
recall = tp / (tp + fn + 1e-7)
f1 = 2 * (precision * recall) / (precision + recall + 1e-7)
accuracy = (tp + tn) / (tp + fp + fn + tn + 1e-7)
specificity = tn / (tn + fp + 1e-7)
dice = dice_coefficient(y_test, pred_probs).numpy()

# =====================================================
# 10. PRINT RESULTS
# =====================================================

print("\n" + "="*60)
print("FINAL EVALUATION – RECORD-LEVEL TEST")
print("="*60)
print(f"Test Records : {TEST_RECORDS}")
print(f"Precision    : {precision.numpy()*100:.2f}%")
print(f"Recall       : {recall.numpy()*100:.2f}%")
print(f"Specificity  : {specificity.numpy()*100:.2f}%")
print(f"F1 Score     : {f1.numpy()*100:.2f}%")
print(f"Accuracy     : {accuracy.numpy()*100:.2f}%")
print(f"Dice         : {dice*100:.2f}%")
print("="*60)

# =====================================================
# 11. SAVE RESULTS
# =====================================================

results = {
    "records": TEST_RECORDS,
    "precision": float(precision.numpy()),
    "recall": float(recall.numpy()),
    "specificity": float(specificity.numpy()),
    "f1": float(f1.numpy()),
    "accuracy": float(accuracy.numpy()),
    "dice": float(dice),
    "tp": int(tp.numpy()),
    "fp": int(fp.numpy()),
    "tn": int(tn.numpy()),
    "fn": int(fn.numpy())
}

with open("final_record_level_evaluation.json", "w") as f:
    json.dump(results, f, indent=2)

print("\n✓ Results saved to final_record_level_evaluation.json")
print("✓ EVALUATION COMPLETE – SAFE & VALID")
